# Task 29: ColumnTransformer Preprocessing

Build a reproducible preprocessing and classification workflow for mixed numerical and categorical data. A `ColumnTransformer` applies different transformations to each feature group, and a `Pipeline` keeps those transformations with the model.

## Learning objectives

- Identify numerical and categorical feature columns.
- Impute missing values, scale numerical features, and one-hot encode categorical features.
- Combine the transformations and classifier in a scikit-learn pipeline.
- Evaluate the model on a held-out test split without fitting preprocessing on test data.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42

## 1. Load and inspect the dataset

In [2]:
data_path = Path("titanic.csv")
if not data_path.exists():
    raise FileNotFoundError("titanic.csv not found. Run this notebook from the Task29 folder.")

df = pd.read_csv(data_path)
target = "survived"

print(f"Dataset shape: {df.shape}")
display(df.head())
display(df.isna().sum().rename("missing values").to_frame())

Dataset shape: (891, 9)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,alone
0,0,3,male,22.0,1,0,7.2500,S,False
1,1,1,female,38.0,1,0,71.2833,C,False
2,1,3,female,26.0,0,0,7.9250,S,True
3,1,1,female,35.0,1,0,53.1000,S,False
4,0,3,male,35.0,0,0,8.0500,S,True


,missing values
survived,0
pclass,0
sex,0
age,177
sibsp,0
parch,0
fare,0
embarked,2
alone,0


## 2. Identify feature types

The target is separated before preprocessing. Passenger class, age, family counts, and fare are treated as numerical features. Sex, embarkation port, and the `alone` indicator are treated as categorical features.

In [3]:
X = df.drop(columns=target)
y = df[target]

numeric_features = ["pclass", "age", "sibsp", "parch", "fare"]
categorical_features = ["sex", "embarked", "alone"]

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)
assert set(numeric_features + categorical_features) == set(X.columns)

Numerical features: ['pclass', 'age', 'sibsp', 'parch', 'fare']
Categorical features: ['sex', 'embarked', 'alone']


## 3. Split the data

A stratified split preserves the target proportions. The test data is held out until evaluation; all imputers and the scaler will be fitted through the training pipeline only.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")

Training rows: 712
Test rows: 179


## 4. Build the preprocessing and model pipeline

Numerical values are median-imputed and standardized. Categorical values are filled with their most frequent value and one-hot encoded. `handle_unknown="ignore"` allows the pipeline to process categories that appear only in the test split.

In [5]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000)),
    ]
)

model.fit(X_train, y_train)
print("Pipeline fitted successfully.")

Pipeline fitted successfully.


## 5. Evaluate on the held-out test set

In [6]:
y_pred = model.predict(X_test)

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision (survived)": precision_score(y_test, y_pred, zero_division=0),
    "Recall (survived)": recall_score(y_test, y_pred, zero_division=0),
    "F1 score (survived)": f1_score(y_test, y_pred, zero_division=0),
}
display(pd.Series(metrics, name="Test score").to_frame())

print("Classification report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("Confusion matrix (rows: actual, columns: predicted):")
display(pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=["Actual: did not survive", "Actual: survived"],
    columns=["Predicted: did not survive", "Predicted: survived"],
))

,Test score
Accuracy,0.804469
Precision (survived),0.783333
Recall (survived),0.681159
F1 score (survived),0.728682


Classification report:
              precision    recall  f1-score   support

           0       0.82      0.88      0.85       110
           1       0.78      0.68      0.73        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179

Confusion matrix (rows: actual, columns: predicted):


,Predicted: did not survive,Predicted: survived
Actual: did not survive,97,13
Actual: survived,22,47


## Conclusion

`ColumnTransformer` routes each feature group through its appropriate preprocessing steps, and the enclosing pipeline ensures those steps are learned from training data. The reported metrics describe this particular test split; they are not a guarantee of performance on other data.